# Video Game Market Analysis

Strategic analysis of historical video game sales to identify market success patterns across platforms, genres, regions, and review signals.

## 1. Setup

Imports, project paths, reusable helpers, and plotting configuration.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import preprocess_games, filter_relevant_period
from src.analysis import (
    annual_release_counts,
    platform_sales,
    platform_lifecycle,
    genre_sales,
    review_sales_correlation,
    regional_top,
)
from src.hypothesis import compare_user_scores

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


## 2. Load and Prepare Data

The raw dataset is normalized into a consistent analytical table. User scores marked as `tbd` are treated as missing values, ratings are marked as `Unknown`, and global sales are calculated from regional sales.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'games.csv'
raw_games = pd.read_csv(DATA_PATH)
games = preprocess_games(raw_games)
recent = filter_relevant_period(games, start_year=2012, end_year=2016)

print('Raw shape:', raw_games.shape)
print('Prepared shape:', games.shape)
print('Recent market window:', recent['year_of_release'].min(), '-', recent['year_of_release'].max())
games.head()


## 3. Data Quality Check

This table summarizes column types, missingness, and cardinality after preprocessing.

In [ ]:
quality = pd.DataFrame({
    'dtype': games.dtypes.astype(str),
    'missing': games.isna().sum(),
    'unique': games.nunique(dropna=True),
})
quality


## 4. Release Trend

Release volume over time helps decide which historical periods remain relevant for forward-looking campaign planning.

In [ ]:
release_counts = annual_release_counts(games)

plt.figure(figsize=(12, 5))
sns.lineplot(data=release_counts, x='year_of_release', y='games_released', marker='o')
plt.title('Game releases by year')
plt.xlabel('Release year')
plt.ylabel('Games released')
plt.tight_layout()
plt.show()

release_counts.tail(10)


## 5. Platform Market Structure

Platforms are ranked by global sales to identify historical leaders and understand console-generation dynamics.

In [ ]:
platform_rank = platform_sales(games)
platform_rank.head(10)


In [ ]:
top_platforms = platform_rank.head(10)['platform']
platform_yearly = (
    games[games['platform'].isin(top_platforms)]
    .groupby(['year_of_release', 'platform'], as_index=False)['global_sales']
    .sum()
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=platform_yearly, x='year_of_release', y='global_sales', hue='platform', marker='o')
plt.title('Global sales trend for top platforms')
plt.xlabel('Release year')
plt.ylabel('Global sales, USD millions')
plt.tight_layout()
plt.show()


## 6. Platform Lifecycle

A platform can dominate historically while being irrelevant for a future campaign. Lifecycle metrics help separate legacy strength from current opportunity.

In [ ]:
lifecycle = platform_lifecycle(games)
lifecycle.head(12)


## 7. Recent Market Window

For campaign planning, recent years carry more operational signal than the full historical archive.

In [ ]:
recent_platform_rank = platform_sales(recent)
recent_platform_rank.head(10)


In [ ]:
recent_top = recent_platform_rank.head(8)['platform']
plt.figure(figsize=(12, 5))
sns.boxplot(data=recent[recent['platform'].isin(recent_top)], x='platform', y='global_sales', showfliers=False)
plt.title('Sales distribution by recent leading platforms')
plt.xlabel('Platform')
plt.ylabel('Global sales, USD millions')
plt.tight_layout()
plt.show()


## 8. Review Scores vs Sales

The analysis compares critic and user score relationships with sales for selected platforms. This is not causal proof, but it indicates whether review signals may help prioritization.

In [ ]:
for platform in ['PS4', 'XOne', '3DS']:
    if platform in recent['platform'].unique():
        print(f'\n{platform}')
        print(review_sales_correlation(recent, platform).round(3))


## 9. Genre Performance

Genre-level sales reveal where demand concentrates and where campaign messaging may need different positioning.

In [ ]:
genre_rank = genre_sales(recent)
plt.figure(figsize=(12, 5))
sns.barplot(data=genre_rank, x='genre', y='global_sales')
plt.title('Recent global sales by genre')
plt.xlabel('Genre')
plt.ylabel('Global sales, USD millions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

genre_rank


## 10. Regional Profiles

North America, Europe, and Japan are profiled separately because platform and genre preferences are not globally uniform.

In [ ]:
regions = ['na_sales', 'eu_sales', 'jp_sales']
for region in regions:
    print(f'\nTop platforms by {region}')
    display(regional_top(recent, region, 'platform'))
    print(f'\nTop genres by {region}')
    display(regional_top(recent, region, 'genre'))


## 11. ESRB Rating Profiles

ESRB ratings are compared across regions. Missing or unknown ratings should be interpreted carefully, especially outside North America.

In [ ]:
rating_profiles = {}
for region in ['na_sales', 'eu_sales', 'jp_sales']:
    rating_profiles[region] = (
        recent.groupby('rating')[region]
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
    print(f'\n{region}')
    display(rating_profiles[region])


## 12. Hypothesis Testing

Two comparisons are evaluated using Levene variance diagnostics and independent t-tests:

1. Xbox One vs PC user scores.
2. Action vs Sports user scores.

In [ ]:
xone_pc = compare_user_scores(recent, 'platform', 'XOne', 'PC', alpha=0.05)
action_sports = compare_user_scores(recent, 'genre', 'Action', 'Sports', alpha=0.05)

pd.DataFrame([xone_pc.__dict__, action_sports.__dict__])


## 13. Conclusions

- Platform timing is a major driver of commercial relevance.
- Recent-generation platforms are more useful for campaign planning than legacy platforms.
- Genre performance differs across markets.
- Japan requires a different market lens than North America and Europe.
- Critic scores tend to be more commercially informative than user scores.
- Hypothesis testing adds statistical discipline to market comparisons.